# EuroCropsML: EDA

EuroCropsML is the crop-type benchmark: per-parcel Sentinel-2 time series across
Estonia, Latvia and Portugal, labelled with HCAT crop codes. We use it for
transnational transfer (train Latvia + Portugal, test Estonia).

## Setup

In [ ]:
from pathlib import Path
import glob, json
import numpy as np
import matplotlib.pyplot as plt

# find the repo root by walking up to the folder that has data/
REPO = Path.cwd()
while not (REPO / 'data').exists() and REPO != REPO.parent:
    REPO = REPO.parent
print('repo:', REPO)

EC = REPO / 'data' / 'input' / 'eurocropsml' / 'preprocess'
files = sorted(glob.glob(str(EC / '*.npz')))
print(len(files), 'EuroCropsML parcels')
# country is the filename prefix; HCAT crop code is the last underscore field
country_of = {'EE': 'Estonia', 'LV': 'Latvia', 'PT': 'Portugal'}

## Raw layout of one parcel

Each `.npz` holds `data` `(T, 13)` (13 S2 bands: B1..B12, SCL), `dates` `(T,)`,
and `center` `[lon, lat]`. T varies per parcel (irregular acquisitions).

In [ ]:
path = files[0]
d = np.load(path)
data, dates, center = d['data'].astype(float), d['dates'], d['center']
code = Path(path).stem.split('_')[-1]      # HCAT crop code
country = country_of.get(Path(path).stem[:2], Path(path).stem[:2])
print('file:', Path(path).name)
print('shape:', data.shape, '| timesteps:', len(dates), '| crop HCAT:', code, '| country:', country)
print('first dates:', list(dates[:4]))

## What this one parcel looks like over its season

Bands are at raw reflectance scale (B4 = red, B8 = NIR); we add the NDVI it implies.

In [ ]:
# band order in the npz: B1,B2,B3,B4,B5,B6,B7,B8,B8A,B9,B11,B12,SCL
doy = dates.astype('datetime64[D]')
b4, b8 = data[:, 3], data[:, 7]
ndvi = np.where((b8 + b4) > 0, (b8 - b4) / (b8 + b4), 0)

fig, ax = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
for c, n in zip([1, 2, 3, 7, 10, 11], ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']):
    ax[0].plot(doy, data[:, c], marker='.', label=n)
ax[0].set_title('Sentinel-2 reflectance'); ax[0].set_ylabel('reflectance'); ax[0].legend(ncol=6, fontsize=8)
ax[1].plot(doy, ndvi, marker='.', color='green')
ax[1].set_title('NDVI'); ax[1].set_ylabel('NDVI'); ax[1].axhline(0, color='grey', lw=0.6)
fig.suptitle(f'one {country} parcel  (HCAT {code})')
plt.tight_layout(); plt.show()

## Country mix and irregular sequence lengths

We sample a few thousand filenames (cheap) rather than opening every parcel.

In [ ]:
import collections
rng = np.random.default_rng(0)
sample = [files[i] for i in rng.permutation(len(files))[:6000]]
countries = [country_of.get(Path(p).stem[:2], '?') for p in sample]
lengths = []
for p in sample[:1500]:
    lengths.append(len(np.load(p)['dates']))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
cc = collections.Counter(countries)
ax[0].bar(list(cc), list(cc.values()), color=['#2563eb', '#16a34a', '#f97316'])
ax[0].set_title('parcels by country (sample of 6000)')
ax[1].hist(lengths, bins=30, color='#7c3aed')
ax[1].set_title('timesteps per parcel'); ax[1].set_xlabel('# acquisitions')
plt.tight_layout(); plt.show()

## Crop-type granularity (why we coarsen HCAT)

The full 10-digit HCAT code gives 100+ long-tail classes; the `crop-class` task
truncates it to 6 digits (~36 crop types). Here is the long tail.

In [ ]:
codes = [Path(p).stem.split('_')[-1] for p in sample]
full = collections.Counter(codes)
coarse = collections.Counter(c[:6] for c in codes)
print('full HCAT classes:', len(full), '| coarsened (6-digit):', len(coarse))
top = coarse.most_common(15)
plt.figure(figsize=(11, 4))
plt.bar([k for k, _ in top], [v for _, v in top], color='#0ea5e9')
plt.title('top coarsened crop types (6-digit HCAT)'); plt.xticks(rotation=60); plt.ylabel('parcels')
plt.tight_layout(); plt.show()

## Takeaways

: one sample is an irregular S2-only parcel time series (no native S1 / climate).
: label = HCAT crop code, coarsened to ~36 types for the task.
: country is the holdout group: train Latvia + Portugal, test Estonia.